# Pb-Tb Adjustment Workflow

This notebook demonstrates how the `gliquid` workflow can be used to adjust predictions using experimental measurements:

1. Generate an initial ML-guided prediction.
2. Overlay measured DSC points.
3. Refine liquid parameters using the measured liquidus constraints.

In [1]:
## UNCOMMENT THIS BLOCK FOR GOOGLE COLAB
# !git clone https://github.com/willwerj/gliquid_python.git
# %cd /content/gliquid_python
# !pip install -q .
# from pathlib import Path
# import gliquid.config as cfg
# cfg.set_data_dir(Path("/content/gliquid_python/data").resolve())

In [ ]:
import os
from pathlib import Path
from gliquid.production_model_runner import ProductionModelRunner

# Required for fetching any uncached MP data.
os.environ.setdefault("NEW_MP_API_KEY", "YOUR_API_KEY_HERE")

# Bundle with trained production models + feature metadata.
bundle_dir = str(Path.cwd().parent / 'data' / '20260329_022905')
runner = ProductionModelRunner(bundle_dir)

## 1 - Plot predicted liquidus curve

Use latest ML model ensemble to predict non-ideal mixing parameters for the Pb-Tb system and plot the resulting phase diagram.

In [3]:
from gliquid.binary import BinaryLiquid, BLPlotter

system = "Pb-Tb"
bl = BinaryLiquid.from_cache(system, param_format='comb-exp')

pred_params = runner.predict_system(system) + [0]
bl.update_params(pred_params)
predicted_liquidus_line = [[p[0], p[1]] for p in bl.phases[-1]['points']]  # Keep a copy for later overlay

blp = BLPlotter(bl)
blp.show('pred')   # Predicted phase diagram
blp.show('ch+g')   # DFT convex hull with liquid free-energy overlay

Pb: H_liq = 4774.0 J/mol, S_liq = 7.9474 J/(mol·K), T_fusion = 600.7 K, polymorphs = 1
Tb: H_liq = 15217.0 J/mol, S_liq = 9.4579 J/(mol·K), T_fusion = 1629.0 K, polymorphs = 2

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.


No arguments specified for 't_vals', setting 't_units' to 'K'


Following this initial prediction, this system was identified as a system that our collaborators at Baylor University
were capable of conducting an assessment on using their DSC techniques.

Below is a table of measurements from their assessment of the Pb-Tb system:

## Table 2. DSC runs of the Tb-Pb system and weight % change

| Composition (%Tb: %Pb) | Solidus (±0.5°C) | Liquidus (±0.5°C) | Weight Change (±0.1%) |
|-------------------------|------------------|-------------------|----------------------|
| 10: 90                  | -                | 857.8             | 0.081                |
| 80: 20a                 | 991.4            | 1152.7            | 0.000                |
| 80: 20b                 | 991.1            | 1157.4            | 0.062                |
| 80: 20c                 | 985.8            | 1152.2            | 0.122                |
| 90: 10a                 | 990.0            | 1150.7            | 0.128                |
| 90: 10b                 | 990.7            | ~1154             | 0.079                |
| 90: 10c                 | 981.7            | ~1133.3           | 0.010                |

The letters a,b,c after the compositions are there only to distinguish samples of the same composition. The exponents are to signify samples that were prepared differently while trying to determine the preparation method that provides the most homogenous sample. 

Credit:

Mario A. Plata,<sup>a</sup> J. Streit Smith,<sup>a</sup> and Julia Y. Chan<sup>* a</sup><br>
<sup>a</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States.<br>
<sup>*</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States;<br>
https://orcid.org/0000-0003-4434-2160; Email: Julia_Chan@baylor.edu<br>


In [4]:
# composition wt% Tb, solidus temperature (C), liquidus temperature (C)
measured_points_wtpct = [[10,   None,   857.8],
                         [80,   991.4,  1152.7],
                         [80,   991.1,  1157.4],
                         [80,   985.8,  1152.2],
                         [90,   990.0,  1150.7],
                         [90,   990.7,  1154],
                         [90,   981.7,  1133.3]]

## 2 - Overlay measured points on predicted phase diagram

Convert repeated measurements into averaged solidus/liquidus observations at each composition, then plot the measured points against the initial predicted phase diagram.

In [5]:
def process_measured_points(measured_points_wtpct):
    """
    Average multiple measurements at the same composition.
    Return solidus points, liquidus points, and compositions with no melting.
    """
    s, l, = [], []
    s_dict, l_dict = {}, {}

    for wt_pct, solidus, liquidus in measured_points_wtpct:

        if solidus is not None:
            if wt_pct not in s_dict:
                s.append([wt_pct, 0])
            s_dict[wt_pct] = s_dict.get(wt_pct, []) + [solidus]
        if liquidus is not None:
            if wt_pct not in l_dict:
                l.append([wt_pct, 0])
            l_dict[wt_pct] = l_dict.get(wt_pct, []) + [liquidus]

    for point in s:
        point[1] = round(sum(s_dict[point[0]]) / len(s_dict[point[0]]), 1)
    for point in l:
        point[1] = round(sum(l_dict[point[0]]) / len(l_dict[point[0]]), 1)

    return s, l,

solidus_points, liquidus_points = process_measured_points(measured_points_wtpct)
print("Solidus points:", solidus_points)
print("Liquidus points:", liquidus_points)

Solidus points: [[80, 989.4], [90, 987.5]]
Liquidus points: [[10, 857.8], [80, 1154.1], [90, 1146.0]]


In [6]:
import plotly.graph_objects as go

def add_experimental_traces(figure: go.Figure):
    """Add measured solidus and liquidus markers to a Plotly figure."""

    figure.add_trace(go.Scatter(
            x=[point[0] for point in solidus_points],
            y=[point[1] for point in solidus_points],
            mode='markers',
            marker=dict(color='#ADEBB3', size=14, symbol='triangle-up', line=dict(width=1, color='black')),
            name='Solidus Observed'
    ))

    figure.add_trace(go.Scatter(
            x=[point[0] for point in liquidus_points],
            y=[point[1] for point in liquidus_points],
            mode='markers',
            marker=dict(color='cornflowerblue', size=12, symbol='square', line=dict(width=1, color='black')),
            name='Liquidus Observed'
    ))

fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.show()

## 3 - Refine liquid parameters using measured constraints

We first adjust the solid phases with the convex hull editor, then use the measured liquidus to refine the non-ideal mixing parameters. The `ConvexHullEditor` is initialized directly from the `BinaryLiquid`; its temperature slider shows the Gibbs hull `G = H - T*S`, and `apply()` writes the edited hull and phase list back to the object.

In [ ]:
import numpy as np
from gliquid.binary import _x_vals
from gliquid.hull_editor import ConvexHullEditor

def optimize_predicted_params(bl: BinaryLiquid, known_points: list, predicted_params: list) -> tuple:

    hull_points = np.array([[p['comp'], p['enthalpy']] for p in bl.phases if 'comp' in p])
    bl.eqs['h_hull_interp'] = np.interp(_x_vals[1:-1], hull_points[:, 0], hull_points[:, 1])
    bl.comp_range_fit_lim = 0
    bl.init_error = False
    bl.max_liq_temp = max(v['T_fusion'] for v in bl.component_data.values())
    bl.min_liq_temp = min(v['T_fusion'] for v in bl.component_data.values())
    bl.digitized_liq = [[p[0], p[1]] for p in known_points]

    mae, _, _, _ = bl.calculate_deviation_metrics(ignored_ranges=False, allow_sparse_data=True)
    print(f"Known points (Kelvin): {known_points}")
    print(f"Liquidus MAE from known points: {int(mae)}K")

    print(f"Original predicted parameters: {predicted_params}")
   
    bl.fit_parameters(verbose=True, max_iter=128, check_phase_mismatch=False, allow_sparse_data=True,
                      disable_inv_constrs=True, check_lupis_elliott=True)
    
    print(f"Adjusted parameters: {bl.get_params()}")
    return bl.get_params()


# Edit the solid phases with the convex hull editor (initialized from the BinaryLiquid).
# Lower the TbPb3 formation enthalpy and add a small amount of alpha-Tb solid solubility,
editor = ConvexHullEditor(bl)
editor.modify_entry(editor.find_entry(name='TbPb3'), new_entropy=3.1, units='J/mol')  # = 3.1 J/mol-atom/K
if not any(e['name'] == 'alpha-Tb (x=0.97)' for e in editor.entries):
    editor.add_entry('Pb0.03Tb0.97', -5700, name='alpha-Tb (x=0.97)', units='J/mol')  # = -5.7 kJ/mol-atom
editor.display()   # interactive: inspect the hull and tune values
editor.apply()

print("Solid phase energies (eV/atom):")
for p in bl.phases:
    if 'comp' in p:
        print(f"  {p['name']:<22} Ef={p['enthalpy']/96485:+.3f}  S={p.get('entropy', 0):+.2f}")
print("-" * 30)

# Reformat measured liquidus points to [at. fraction, Kelvin] and add inferred eutectic point for optimization.
measured_liq_points = [[p[0]/100.0, p[1]+273.15] for p in liquidus_points]
avg_comp_wtpct = sum(p[0] for p in solidus_points) / len(solidus_points)
avg_temp_c = sum(p[1] for p in solidus_points) / len(solidus_points)
measured_liq_points.append([avg_comp_wtpct / 100.0, avg_temp_c + 273.15])
measured_liq_points.sort()

adj_params = optimize_predicted_params(bl, measured_liq_points, pred_params)
blp = BLPlotter(bl)
fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.show()

Solid phase energies (eV/atom):
  fcc-Pb                 Ef=+0.000  S=+0.00
  TbPb3                  Ef=-0.336  S=+0.00
  Tb5Pb4                 Ef=-0.542  S=+0.00
  Tb5Pb3                 Ef=-0.543  S=+0.00
  alpha-Tb (x=0.97)      Ef=-0.059  S=+0.00
  alpha-Tb (hcp)         Ef=+0.000  S=+0.00
  beta-Tb (bcc)          Ef=+0.046  S=+2.83
------------------------------
Known points (Kelvin): [[0.1, 1130.9499999999998], [0.8, 1427.25], [0.9, 1419.15]]
Liquidus MAE from known points: 176K
Original predicted parameters: [-154181.125, -18.407392501831055, 55601.482421875, 0]

Maximum composition range fitted: [0.1, 0.9]
Ignored composition ranges: []

Initial triangle for H-S partition constraints: [[-117338.4957735932, -26546.732020483774], [-117338.4957735932, 16933.440402886645], [-72519.17857686648, 16933.440402886645]]
--- Beginning Nelder-Mead optimization ---
Iteration # 0
Best guess: {a: -117338.4957735932, c: 16933.440402886645} f=365.4482327899156
Height of triangle = 44819.317196